# IR Assignment 2 — Notebook 04: Pipeline Retrieval (R3, R4, R5)

Implements the **retrieve-then-rerank** pipeline from the proposal (Figure 1, Table 1):

```
Query → BM25F (Stage 1) → Top-100 candidates → Dense re-rank (Stage 2) → Top-10
```

- **R3** — BM25F candidates re-ranked by dense bi-encoder similarity
- **R4** — Weighted sum: `score = α × norm(bm25f) + (1−α) × norm(dense)` on the same candidates
- **R5** — RRF: `score = 1/(k + bm25f_rank) + 1/(k + dense_rank)` on the same candidates

Dense scores only the **BM25F top-N candidates** — not the full corpus.
No FAISS required for re-ranking (only ~100 candidates to score per query).

In [2]:
!pip install rank_bm25

In [1]:
from pathlib import Path
import re
import json

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi

## 1 — Paths & Settings


In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
INTERIM_DIR  = PROJECT_ROOT / "data" / "interim"
RESULTS_DIR  = PROJECT_ROOT / "results" / "fusion"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PARSED_PATH = INTERIM_DIR / "parsed_patents.jsonl"

# BM25F field weights (must match Notebook 02)
FIELD_WEIGHTS = {"title": 3.0, "abstract": 2.0, "claims": 1.5, "description": 1.0}
FIELDS        = list(FIELD_WEIGHTS.keys())

TOP_K      = 10    # final results returned
POOL_SIZE  = 200   # BM25F candidates fed into dense re-ranker

print("PARSED_PATH exists:", PARSED_PATH.exists())

PARSED_PATH exists: True


## 2 — Load Data


In [3]:
docs_df = pd.read_json(PARSED_PATH, lines=True)
docs_df["ucid"] = docs_df["ucid"].astype(str)
for field in FIELDS:
    docs_df[field] = docs_df[field].fillna("")

TOKEN_RE = re.compile(r"[A-Za-z0-9]+")
def tokenize(text: str):
    return TOKEN_RE.findall(text.lower())

print(f"Loaded {len(docs_df):,} patents")
docs_df[["ucid","title"]].head(3)


Loaded 20,000 patents


,ucid,title
0,WO-1979000001-A1,APPARATUS FOR DETERMINING THE EPIDERMIC GROUP ...
1,WO-1979000002-A1,IMPROVEMENTS RELATING TO MEMBRANE ELECTROPHORESIS
2,WO-1979000005-A1,IMPROVED CLUTCH-BRAKE SYSTEM FOR ROTARY MOWER


## 3 — Build BM25F Index (R2)


In [4]:
for field in FIELDS:
    docs_df[f"tokens_{field}"] = docs_df[field].map(tokenize)

bm25_indexes = {}
for field in FIELDS:
    bm25_indexes[field] = BM25Okapi(docs_df[f"tokens_{field}"].tolist())
    print(f"BM25 index built for '{field}'")


BM25 index built for 'title'
BM25 index built for 'abstract'
BM25 index built for 'claims'
BM25 index built for 'description'


In [5]:
def search_bm25f(query_text: str, top_k: int = POOL_SIZE, exclude_ucid: str = None):
    """Returns a DataFrame with columns: ucid, bm25f_score, rank (top_k results)."""
    tokens = tokenize(query_text)
    if not tokens:
        raise ValueError("Empty query after tokenisation.")

    scores = np.zeros(len(docs_df))
    for field, weight in FIELD_WEIGHTS.items():
        scores += weight * bm25_indexes[field].get_scores(tokens)

    result_df = docs_df[["ucid", "title"]].copy()
    result_df["bm25f_score"] = scores

    if exclude_ucid:
        result_df = result_df[result_df["ucid"] != exclude_ucid]

    result_df = result_df.sort_values("bm25f_score", ascending=False).head(top_k).reset_index(drop=True)
    result_df.insert(0, "rank", range(1, len(result_df) + 1))
    return result_df


print("search_bm25f defined")


search_bm25f defined


## 4 — Load Dense Re-ranker (R3)

In [6]:
from sentence_transformers import SentenceTransformer
import torch

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

# Switching from anferico/bert-for-patents (MLM pre-training only, poor retrieval)
# to all-MiniLM-L6-v2 (contrastive fine-tuning on 1B+ pairs, designed for retrieval)
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME, device=DEVICE)

print(f"Loaded '{MODEL_NAME}' on device: {DEVICE}")
print(f"Max sequence length: {model.max_seq_length}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded 'sentence-transformers/all-MiniLM-L6-v2' on device: cuda
Max sequence length: 256


In [7]:
TAG_SPACE_RE = re.compile(r"\s+")

def create_overlapping_chunks(text, title, chunk_size=300, overlap=50):
    """Identical to Notebook 03 — breaks text into overlapping chunks with title context."""
    if not text or not str(text).strip():
        return []
    words = str(text).split()
    chunks = []
    step_size = chunk_size - overlap
    for i in range(0, len(words), step_size):
        chunk_words = words[i: i + chunk_size]
        chunk_text  = " ".join(chunk_words)
        chunks.append(f"Title: {title} | Text: {chunk_text}")
        if i + chunk_size >= len(words):
            break
    return chunks


def rerank_with_dense(query_text: str, candidates_df: pd.DataFrame) -> dict:
    """
    Re-ranks BM25F candidates using the same chunking + MaxP strategy as Notebook 03.

    Stage 2 pipeline:
      - Full text (abstract + description + claims) is chunked (300w / 50w overlap)
      - All chunks for the 100 candidates are embedded in one batch
      - MaxP aggregation: each patent's score = its highest-scoring chunk
    """
    query_emb = model.encode([query_text], normalize_embeddings=True)

    ucids       = candidates_df["ucid"].tolist()
    all_chunks  = []   # flat list of chunk texts
    chunk_ucids = []   # which ucid each chunk belongs to

    for ucid in ucids:
        row = docs_df[docs_df["ucid"] == ucid]
        if row.empty:
            continue
        r = row.iloc[0]
        title_text = (r["title"] or "Unknown Title").strip()
        full_text  = TAG_SPACE_RE.sub(
            " ",
            f"{r['abstract']} {r['description']} {r['claims']}"
        ).strip()
        chunks = create_overlapping_chunks(full_text, title=title_text)
        if not chunks:
            chunks = [f"Title: {title_text} | Text: "]   # fallback for empty patents
        all_chunks.extend(chunks)
        chunk_ucids.extend([ucid] * len(chunks))

    # Encode all chunks in one batch
    chunk_embs = model.encode(all_chunks, normalize_embeddings=True, show_progress_bar=False)

    # Cosine similarity: query vs every chunk
    chunk_scores = (query_emb @ chunk_embs.T).flatten()

    # MaxP: keep the best chunk score per patent
    patent_scores = {}
    for ucid, score in zip(chunk_ucids, chunk_scores):
        if ucid not in patent_scores or score > patent_scores[ucid]:
            patent_scores[ucid] = float(score)

    return patent_scores


print("rerank_with_dense defined (chunking + MaxP, consistent with Notebook 03)")

rerank_with_dense defined (chunking + MaxP, consistent with Notebook 03)


## 5 — Pipeline Functions (R3, R4, R5)

All three systems share the **same BM25F top-100 candidate pool** — dense scores only those 100 documents.

In [8]:
def search_r3(query_text: str, top_k: int = TOP_K, exclude_ucid: str = None):
    """
    R3 — BM25F → Dense re-rank (retrieve-then-rerank pipeline).
    Stage 1: BM25F retrieves top POOL_SIZE candidates.
    Stage 2: Dense bi-encoder re-ranks those candidates by similarity.
    Returns top_k results sorted by dense score.
    """
    candidates = search_bm25f(query_text, top_k=POOL_SIZE, exclude_ucid=exclude_ucid)
    dense_scores = rerank_with_dense(query_text, candidates)

    ranked = sorted(dense_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    rows = []
    for rank, (ucid, dense_score) in enumerate(ranked, 1):
        row = docs_df[docs_df["ucid"] == ucid]
        title = row.iloc[0]["title"] if not row.empty else ""
        bm25f_score = candidates.loc[candidates["ucid"] == ucid, "bm25f_score"].values
        rows.append({
            "rank":         rank,
            "ucid":         ucid,
            "dense_score":  round(dense_score, 6),
            "bm25f_score":  round(float(bm25f_score[0]), 4) if len(bm25f_score) else 0.0,
            "title":        title,
        })
    return pd.DataFrame(rows)


print("search_r3 defined")

search_r3 defined


In [9]:
def fuse_r4(query_text: str, alpha: float = 0.5, top_k: int = TOP_K,
            exclude_ucid: str = None):
    """
    R4 — Weighted sum fusion on BM25F candidate pool.
    Stage 1: BM25F retrieves top POOL_SIZE candidates.
    Stage 2: Dense re-ranks the same candidates.
    score = alpha * norm(bm25f) + (1 - alpha) * norm(dense)
    Both score lists are min-max normalised to [0, 1] before combining.
    """
    candidates   = search_bm25f(query_text, top_k=POOL_SIZE, exclude_ucid=exclude_ucid)
    dense_scores = rerank_with_dense(query_text, candidates)

    bm25f_scores = dict(zip(candidates["ucid"], candidates["bm25f_score"]))

    def minmax(d):
        vals = list(d.values())
        lo, hi = min(vals), max(vals)
        rng = hi - lo if hi > lo else 1.0
        return {k: (v - lo) / rng for k, v in d.items()}

    bm25f_norm = minmax(bm25f_scores)
    dense_norm = minmax(dense_scores)

    # Every ucid is in both dicts (same candidate pool)
    combined = {
        ucid: alpha * bm25f_norm[ucid] + (1 - alpha) * dense_norm[ucid]
        for ucid in bm25f_norm
    }

    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)[:top_k]

    rows = []
    for rank, (ucid, score) in enumerate(ranked, 1):
        row = docs_df[docs_df["ucid"] == ucid]
        title = row.iloc[0]["title"] if not row.empty else ""
        rows.append({
            "rank":         rank,
            "ucid":         ucid,
            "r4_score":     round(score, 6),
            "bm25f_score":  round(bm25f_scores[ucid], 4),
            "dense_score":  round(dense_scores[ucid], 4),
            "title":        title,
        })
    return pd.DataFrame(rows)


print(f"fuse_r4 defined (default α=0.5)")

fuse_r4 defined (default α=0.5)


In [10]:
def fuse_r5(query_text: str, k: int = 60, top_k: int = TOP_K,
            exclude_ucid: str = None):
    """
    R5 — Reciprocal Rank Fusion (RRF) on BM25F candidate pool.
    Stage 1: BM25F retrieves top POOL_SIZE candidates.
    Stage 2: Dense re-ranks the same candidates.
    score = 1/(k + rank_bm25f) + 1/(k + rank_dense)
    No score normalisation needed — only rank positions matter.
    """
    candidates   = search_bm25f(query_text, top_k=POOL_SIZE, exclude_ucid=exclude_ucid)
    dense_scores = rerank_with_dense(query_text, candidates)

    bm25f_ranks = dict(zip(candidates["ucid"], candidates["rank"]))

    # Rank dense scores (highest score = rank 1)
    dense_ranked = sorted(dense_scores.items(), key=lambda x: x[1], reverse=True)
    dense_ranks  = {ucid: i + 1 for i, (ucid, _) in enumerate(dense_ranked)}

    rrf_scores = {
        ucid: 1.0 / (k + bm25f_ranks[ucid]) + 1.0 / (k + dense_ranks[ucid])
        for ucid in bm25f_ranks  # same candidate pool
    }

    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    rows = []
    for rank, (ucid, score) in enumerate(ranked, 1):
        row = docs_df[docs_df["ucid"] == ucid]
        title = row.iloc[0]["title"] if not row.empty else ""
        rows.append({
            "rank":        rank,
            "ucid":        ucid,
            "rrf_score":   round(score, 6),
            "bm25f_rank":  bm25f_ranks[ucid],
            "dense_rank":  dense_ranks[ucid],
            "title":       title,
        })
    return pd.DataFrame(rows)


print(f"fuse_r5 defined (default k=60)")

fuse_r5 defined (default k=60)


## 6 — Demo: Free-Text Query (R2 vs R3 vs R4 vs R5)


In [11]:
demo_query = "internal combustion engine fuel injection system"

print(f"Query: '{demo_query}'")
print("=" * 70)

r2 = search_bm25f(demo_query, top_k=TOP_K)
r3 = search_r3(demo_query, top_k=TOP_K)
r4 = fuse_r4(demo_query, alpha=0.5, top_k=TOP_K)
r5 = fuse_r5(demo_query, k=60, top_k=TOP_K)

print("\n--- R2: BM25F (Stage 1 only) ---")
print(r2[["rank", "ucid", "bm25f_score", "title"]].to_string(index=False))

print("\n--- R3: BM25F → Dense Re-rank ---")
print(r3[["rank", "ucid", "dense_score", "bm25f_score", "title"]].to_string(index=False))

print("\n--- R4: Weighted Fusion (α=0.5) on BM25F candidates ---")
print(r4[["rank", "ucid", "r4_score", "bm25f_score", "dense_score", "title"]].to_string(index=False))

print("\n--- R5: RRF (k=60) on BM25F candidates ---")
print(r5[["rank", "ucid", "rrf_score", "bm25f_rank", "dense_rank", "title"]].to_string(index=False))

Query: 'internal combustion engine fuel injection system'

--- R2: BM25F (Stage 1 only) ---
 rank             ucid  bm25f_score                                                                                                                                                                title
    1 WO-2000011334-A1   208.831839                                                                                                                        COMPRESSED AIR ASSISTED FUEL INJECTION SYSTEM
    2 WO-2000011343-A1   194.154223                                                                                                                          REGULATION METHOD FOR FUEL INJECTION SYSTEM
    3 WO-2000011337-A1   188.872840                                                                                                                                 FUEL INJECTION SYSTEM CONTROL METHOD
    4 WO-2000020745-A1   184.002990                                                                     

## 7 — R4: Tune α (Sparse vs Dense Weight)


In [12]:
alphas = [0.2, 0.4, 0.5, 0.6, 0.8]

# Pre-compute candidates and dense scores once — only alpha varies
_candidates   = search_bm25f(demo_query, top_k=POOL_SIZE)
_dense_scores = rerank_with_dense(demo_query, _candidates)
_bm25f_scores = dict(zip(_candidates["ucid"], _candidates["bm25f_score"]))

def minmax(d):
    vals = list(d.values())
    lo, hi = min(vals), max(vals)
    rng = hi - lo if hi > lo else 1.0
    return {k: (v - lo) / rng for k, v in d.items()}

_bm25f_norm = minmax(_bm25f_scores)
_dense_norm  = minmax(_dense_scores)

print(f"Query: '{demo_query}'\n")
print(f"{'alpha':>6}  {'Top-3 UCIDs'}")
print("-" * 70)
for alpha in alphas:
    combined = {
        ucid: alpha * _bm25f_norm[ucid] + (1 - alpha) * _dense_norm[ucid]
        for ucid in _bm25f_norm
    }
    top3 = sorted(combined, key=combined.get, reverse=True)[:3]
    print(f"  {alpha:.1f}    {' | '.join(top3)}")

Query: 'internal combustion engine fuel injection system'

 alpha  Top-3 UCIDs
----------------------------------------------------------------------
  0.2    WO-2000011334-A1 | WO-2000011337-A1 | WO-2000029743-A1
  0.4    WO-2000011334-A1 | WO-2000011337-A1 | WO-2000011343-A1
  0.5    WO-2000011334-A1 | WO-2000011337-A1 | WO-2000011343-A1
  0.6    WO-2000011334-A1 | WO-2000011337-A1 | WO-2000011343-A1
  0.8    WO-2000011334-A1 | WO-2000011343-A1 | WO-2000011337-A1


## 8 — Save Results


In [13]:
r3_path = RESULTS_DIR / "r3_rerank_results.csv"
r4_path = RESULTS_DIR / "r4_fusion_results.csv"
r5_path = RESULTS_DIR / "r5_rrf_results.csv"

r3.to_csv(r3_path, index=False)
r4.to_csv(r4_path, index=False)
r5.to_csv(r5_path, index=False)

config = {
    "fusion_query":  demo_query,
    "field_weights": FIELD_WEIGHTS,
    "pool_size":     POOL_SIZE,
    "top_k":         TOP_K,
    "dense_model":   MODEL_NAME,
    "r3_method":     "bm25f_top100 -> chunking_maxp_rerank",
    "r4_alpha":      0.5,
    "r5_k":          60,
}
(RESULTS_DIR / "fusion_config.json").write_text(json.dumps(config, indent=2))

print(f"Model:     {MODEL_NAME}")
print(f"R3 → {r3_path}")
print(f"R4 → {r4_path}")
print(f"R5 → {r5_path}")

Model:     sentence-transformers/all-MiniLM-L6-v2
R3 → /content/results/fusion/r3_rerank_results.csv
R4 → /content/results/fusion/r4_fusion_results.csv
R5 → /content/results/fusion/r5_rrf_results.csv
